# 03 - Next Week Prediction

7-day forecast.


## Setup

In [ ]:
import sys, json
sys.path.insert(0, r"D:\\Projects\\done\\eth_bot\\eth_bot_v2")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from analysis.loader import load_ohlcv
from analysis.indicators import add_all

pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (12, 4)
OUT = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "reports" / "forecasts"
OUT.mkdir(parents=True, exist_ok=True)
FEATURES = ["ret_1","ret_5","ret_20","vol_20","rsi_14","macd_hist","atr_14","sma_50","sma_200"]

def build(timeframe="1d", horizon=1):
    df = load_ohlcv(timeframe=timeframe)
    df = add_all(df).copy()
    df["ret_1"]  = df["close"].pct_change(1)
    df["ret_5"]  = df["close"].pct_change(5)
    df["ret_20"] = df["close"].pct_change(20)
    df["vol_20"] = df["ret_1"].rolling(20).std()
    df["target"] = df["close"].pct_change(horizon).shift(-horizon)
    return df.dropna()

def metrics(y, p):
    return {"rmse": float(np.sqrt(((y-p)**2).mean())),
            "mae":  float(np.abs(y-p).mean()),
            "dir_accuracy": float(((p>0)==(y>0)).mean())}

## 1. Direct 7-day model

In [ ]:
df = build(horizon=7)
train = df.iloc[:-250]; test = df.iloc[-250:]
model = Ridge(alpha=1.0).fit(train[FEATURES].values, train["target"].values)
pred = model.predict(test[FEATURES].values); y = test["target"].values
print(json.dumps(metrics(y, pred), indent=2))

r = float(model.predict(df[FEATURES].iloc[[-1]].values)[0])
last = float(df["close"].iloc[-1]); rs = float((y-pred).std())
lo, hi = last*(1+r-1.28*rs), last*(1+r+1.28*rs)
print(f"7d pred ret: {r*100:+.4f}%   close: {last*(1+r):.2f}")
print(f"80% interval: [{lo:.2f}, {hi:.2f}]")
(OUT/"next_week_direct.json").write_text(json.dumps(
    {"as_of": str(df.index[-1].date()), "horizon_days": 7,
     "last_close": last, "week_ret": r, "week_close": last*(1+r),
     "p10": lo, "p90": hi}, indent=2))

## 2. Chained daily

In [ ]:
df1 = build(horizon=1)
model1 = Ridge(alpha=1.0).fit(df1[FEATURES].values, df1["target"].values)
sim = df1.copy(); daily = []
for _ in range(7):
    r1 = float(model1.predict(sim[FEATURES].iloc[[-1]].values)[0])
    daily.append(r1)
    new = sim.iloc[[-1]].copy()
    new.index = [sim.index[-1] + pd.Timedelta(days=1)]
    new["target"] = np.nan
    sim = pd.concat([sim, new])
chain = (1 + pd.Series(daily)).prod() - 1
print("chained 7d:", f"{chain*100:+.4f}%")
(OUT/"next_week_chain.json").write_text(json.dumps(
    {"chained_7d_ret": float(chain)}, indent=2))